In [2]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
from gene_thesaurus import GeneThesaurus

In [3]:
MODELS_PATH = "/home/mariya/Data/pallas/models/gex"
X_test_path = os.path.join(MODELS_PATH, "merged.test.cal.features.csv")
y_test_path = os.path.join(MODELS_PATH, "merged.test.cal.labels.csv")

random_seed = 42
rf_model_path = os.path.join(MODELS_PATH, "rf_model.pkl")
selected_features_path = os.path.join(MODELS_PATH, "selected_features.txt")
pipeline_path = os.path.join(MODELS_PATH, "pipeline.pkl")
hyperparams_path = os.path.join(MODELS_PATH, "hyperparams.pkl")

In [5]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_test, y_test

(                    ENSG00000000003  ENSG00000001036  ENSG00000001561  \
 id                                                                      
 16-017                    -1.182590         2.589817         2.208578   
 16-094                    -0.855174         3.029173         6.052810   
 16-106                    -1.034587         4.652130         5.741091   
 16-158                    -0.966538         4.821839         2.514669   
 16-176                     2.449653         2.157475         2.389336   
 ...                             ...              ...              ...   
 TARGET-21-PARLSL-D        -1.219101         4.896004         1.342448   
 TCGA-AB-2913              -1.219101         3.893187         5.941705   
 TCGA-AB-2919              -1.219101         4.352004         4.352621   
 TCGA-AB-2924              -1.219101         3.179345         2.505094   
 TCGA-AB-3011              -1.219101         4.502346         4.066546   
 
                     ENSG00000001617

In [6]:
print("\ny_test class distribution:")
print(y_test.value_counts())


y_test class distribution:
class                   
B-ALL, high hyperdiploid    167
AML, KMT2A-r                156
B-ALL, ETV6::RUNX1          139
B-ALL, BCR::ABL1-like       112
AML, RUNX1::RUNX1T1         108
AML, NPM1alt                 68
B-ALL, KMT2A-r               66
B-ALL, PAX5alt               61
T-ALL                        53
AML, CBFB::MYH11             50
B-ALL, DUX4-r                49
B-ALL, BCR::ABL1             42
B-ALL, TCF3::PBX1            38
B-ALL, hypodiploid           37
AML, CEBPAalt                34
AML, NUP98-r                 29
B-ALL, ZNF384-r              24
B-ALL, ETV6::RUNX1-like      19
B-ALL, MEF2D-r               17
B-ALL, iAMP21                15
AML, DEK::NUP214             13
AML, CBFA2T3::GLIS2          11
B-ALL, PAX5 P80R             11
Non-leukemic                  7
AML, APL-PML::RARA            6
B-ALL, NUTM1-r                5
B-ALL, MYC-r                  5
AML, KAT6A-r                  4
B-ALL, HLF-r                  3
AML, ETV6-r        

In [7]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_test

array(['B-ALL, PAX5alt', 'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1', ...,
       'AML, NPM1alt', 'AML, NPM1alt', 'AML, NPM1alt'],
      shape=(1356,), dtype=object)

In [8]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features from text file
selected_features = pd.read_csv(selected_features_path, header=None)[0].tolist()

# Use selected features for test set
y_pred_final = final_rf.predict(X_test)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test.shape}")

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Number of selected features: 3923
Shape of X_test_selected: (1356, 3923)


In [9]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         6
     AML, CBFA2T3::GLIS2      0.900     0.818     0.857        11
        AML, CBFB::MYH11      0.980     0.980     0.980        50
           AML, CEBPAalt      0.943     0.971     0.957        34
        AML, DEK::NUP214      0.929     1.000     0.963        13
             AML, ETV6-r      0.667     0.667     0.667         3
           AML, FUS::ERG      1.000     0.500     0.667         2
            AML, KAT6A-r      1.000     0.500     0.667         4
            AML, KMT2A-r      0.901     0.929     0.915       156
            AML, NPM1alt      0.938     0.897     0.917        68
            AML, NUP98-r      0.833     0.862     0.847        29
     AML, RUNX1::RUNX1T1      1.000     0.991     0.995       108
        B-ALL, BCR::ABL1      0.929     0.619     0.743        42
   B-ALL, BCR::ABL

In [10]:
# ----------------------
# Extract top cpg sites from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene Symbol"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
                 Gene  Importance Gene Symbol
3067  ENSG00000184507    0.029388       NUTM1
317   ENSG00000079102    0.025635     RUNX1T1
1730  ENSG00000145794    0.019836      MEGF10
493   ENSG00000101115    0.017745       SALL4
3632  ENSG00000248329    0.016787       APELA
3699  ENSG00000259803    0.012556    SLC22A31
2267  ENSG00000164100    0.011442       NDST3
2025  ENSG00000157680    0.011384        DGKI
626   ENSG00000105642    0.011167       KCNN1
3081  ENSG00000185046    0.011096      ANKS1B
715   ENSG00000108511    0.010603       HOXB6
1071  ENSG00000124302    0.010031       CHST8
3352  ENSG00000197705    0.009977      KLHL14
972   ENSG00000120075    0.008896       HOXB5
2455  ENSG00000168004    0.008786      PLAAT5
1668  ENSG00000143995    0.008637       MEIS1
909   ENSG00000116703    0.008221         PDC
1104  ENSG00000125821    0.007942        DTD1
1657  ENSG00000143816    0.007469       WNT9A
327   ENSG00000080546    0.0

In [11]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 0 out of 3923

Bottom 20 genes by importance (final model):
                 Gene    Importance      Gene Symbol
2237  ENSG00000163705  4.825921e-06         FANCD2OS
3692  ENSG00000259288  4.815935e-06       BUB1B-PAK6
3481  ENSG00000205669  4.677257e-06            ACOT6
2446  ENSG00000167766  4.665393e-06            ZNF83
1774  ENSG00000147378  4.626656e-06            FATE1
3186  ENSG00000187754  4.593492e-06             SSX7
3766  ENSG00000269096  4.396347e-06           CT45A3
3292  ENSG00000196588  4.314004e-06            MRTFA
3880  ENSG00000278619  4.158926e-06             MRM1
947   ENSG00000118702  4.042164e-06             GHRH
3497  ENSG00000206535  3.607471e-06             LNP1
3480  ENSG00000205643  3.479824e-06            CDPF1
2404  ENSG00000166819  3.336136e-06            PLIN1
2205  ENSG00000163322  3.105085e-06         ABRAXAS1
1495  ENSG00000138207  2.502533e-06             RBP4
3676  ENSG00000257184  2.379288e-06     HOXA10-HOX